# Web Scraping data
v2
10/05/2026

In [1]:
import asyncio
import json
import re
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.robotparser import RobotFileParser
from urllib.parse import urlparse
 
import httpx
import pandas as pd
from bs4 import BeautifulSoup

In [6]:
# Configuration
EXCEL_FILE = Path("/Users/anamoura/Desktop/SMAnalictics/DataExtraction/Bieberchella_urls_v2.xlsx")

def load_urls_from_excel(excel_path: Path) -> list[str]:
    """Load URLs from an Excel file, auto-detecting the URL column."""
    if not excel_path.exists():
        raise FileNotFoundError(f"File not found: {excel_path}")

    df = pd.read_excel(excel_path)

    # Auto-detect URL column
    url_column = None
    for col in df.columns:
        if col.strip().lower() in ("url", "link", "links", "urls", "href"):
            url_column = col
            break

    if url_column is None:
        raise ValueError(
            f"Could not find a URL column.\n"
            f"Columns found: {list(df.columns)}\n"
            "Rename your column to 'url', 'link', or 'href'."
        )

    urls = (
        df[url_column]
        .dropna()
        .astype(str)
        .str.strip()
        .pipe(lambda s: s[s.str.startswith("http")])
        .unique()
        .tolist()
    )
    return urls

In [7]:
# Load URLs
URLS = load_urls_from_excel(EXCEL_FILE)
print(f'📂 Loaded {len(URLS)} URLs from "{EXCEL_FILE}"')
for u in URLS[:5]:
    print(f'  {u}')

📂 Loaded 200 URLs from "/Users/anamoura/Desktop/SMAnalictics/DataExtraction/Bieberchella_urls_v2.xlsx"
  https://www.yahoo.com/entertainment/music/article/justin-bieber-is-headlining-coachella-this-month-how-the-pop-star-is-forging-a-new-path-163753630.html
  https://www.theguardian.com/music/2026/apr/10/coachella-2026-justin-bieber-sabrina-carpenter-karol-g
  https://www.rollingstone.com/music/music-news/justin-bieber-coachella-sneak-preview-1235540010/
  https://pagesix.com/2026/04/04/celebrity-news/justin-biebers-got-a-lot-riding-on-coachella-comeback/
  https://eu.desertsun.com/story/entertainment/coachella/2026/04/03/justin-bieber-loves-palm-springs-but-it-doesnt-always-love-him-back-recap-of-infamous-moments/89319620007/


In [9]:
URLS = load_urls_from_excel(EXCEL_FILE)
 
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
 
JSONL_FILE = OUTPUT_DIR / "raw_articles.jsonl"
PARQUET_FILE = OUTPUT_DIR / "articles.parquet"
FAILED_FILE = OUTPUT_DIR / "failed_urls.txt"
CHECKPOINT_FILE = OUTPUT_DIR / "checkpoint.json"
 
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}
 
CONCURRENCY = 2          # parallel async requests
DELAY_BETWEEN = 10      # seconds between requests (be polite)
PLAYWRIGHT_TIMEOUT = 25  # seconds
HTTP_TIMEOUT = 20
 

In [10]:
# ─── Helpers ──────────────────────────────────────────────────────────────────
 
def load_checkpoint() -> set:
    if CHECKPOINT_FILE.exists():
        data = json.loads(CHECKPOINT_FILE.read_text())
        return set(data.get("done", []))
    return set()
 
def save_checkpoint(done: set):
    CHECKPOINT_FILE.write_text(json.dumps({"done": list(done)}))
 
def append_jsonl(record: dict):
    with JSONL_FILE.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
 
def is_allowed_by_robots(url: str) -> bool:
    try:
        parsed = urlparse(url)
        robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
        rp = RobotFileParser()
        rp.set_url(robots_url)
        rp.read()
        return rp.can_fetch("*", url)
    except Exception:
        return True  # if robots.txt unreachable, assume allowed
 
def parse_html(html: str, url: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")
 
    # Remove noise
    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()
 
    title = soup.title.string.strip() if soup.title else None
 
    # Try to find article body
    article = (
        soup.find("article")
        or soup.find(attrs={"class": re.compile(r"article|content|body|story", re.I)})
        or soup.find("main")
        or soup.body
    )
    text = article.get_text(separator=" ", strip=True) if article else ""
    text = re.sub(r"\s+", " ", text)[:10000]  # cap at 10k chars
 
    # Meta tags
    def meta(name):
        tag = soup.find("meta", attrs={"name": name}) or soup.find("meta", attrs={"property": name})
        return tag.get("content", "").strip() if tag else None
 
    return {
        "url": url,
        "domain": urlparse(url).netloc,
        "title": title,
        "description": meta("description") or meta("og:description"),
        "author": meta("author") or meta("article:author"),
        "published_date": meta("article:published_time") or meta("og:article:published_time"),
        "keywords": meta("keywords"),
        "text": text,
        "text_length": len(text),
        "scraped_at": datetime.now(timezone.utc).isoformat(),
        "method": None,   # filled by caller
        "status": None,   # filled by caller
        "error": None,
    }
 

In [11]:
# ─── Phase 1: Classification ───────────────────────────────────────────────────
 
async def classify_urls(urls: list[str]) -> dict:
    """Quick HEAD requests to classify each URL before full scraping."""
    results = {"ok": [], "blocked": [], "robots_blocked": [], "error": []}
 
    print("\n📋 Phase 1: Classifying URLs...")
 
    async with httpx.AsyncClient(headers=HEADERS, follow_redirects=True, timeout=10) as client:
        for url in urls:
            if not is_allowed_by_robots(url):
                results["robots_blocked"].append(url)
                print(f"  🤖 robots.txt blocked: {url}")
                continue
            try:
                r = await client.head(url)
                if r.status_code in (200, 301, 302):
                    results["ok"].append(url)
                    print(f"  ✅ {r.status_code} {url}")
                elif r.status_code in (403, 401, 429):
                    results["blocked"].append(url)
                    print(f"  🚫 {r.status_code} {url}")
                else:
                    results["error"].append(url)
                    print(f"  ⚠️  {r.status_code} {url}")
            except Exception as e:
                results["error"].append(url)
                print(f"  ❌ error: {url} — {e}")
            await asyncio.sleep(0.2)
 
    print(f"\n  Summary → OK: {len(results['ok'])} | Blocked: {len(results['blocked'])} | "
          f"robots.txt: {len(results['robots_blocked'])} | Error: {len(results['error'])}")
    return results
 

In [12]:
# ─── Phase 2: Async HTTP Scraping ─────────────────────────────────────────────
 
async def scrape_http(url: str, client: httpx.AsyncClient, semaphore: asyncio.Semaphore) -> dict:
    async with semaphore:
        try:
            r = await client.get(url, timeout=HTTP_TIMEOUT)
            record = parse_html(r.text, url)
            record["method"] = "httpx"
            record["status"] = r.status_code
            return record
        except Exception as e:
            return {
                "url": url,
                "domain": urlparse(url).netloc,
                "title": None, "description": None, "author": None,
                "published_date": None, "keywords": None, "text": None,
                "text_length": 0,
                "scraped_at": datetime.now(timezone.utc).isoformat(),
                "method": "httpx", "status": None, "error": str(e),
            }
 
async def run_http_phase(urls: list[str], done: set) -> tuple[list, list]:
    pending = [u for u in urls if u not in done]
    if not pending:
        print("\n⏭️  All HTTP URLs already done (checkpoint).")
        return [], []
 
    print(f"\n🌐 Phase 2: HTTP scraping {len(pending)} URLs...")
    semaphore = asyncio.Semaphore(CONCURRENCY)
    succeeded, failed = [], []
 
    async with httpx.AsyncClient(headers=HEADERS, follow_redirects=True) as client:
        tasks = [scrape_http(u, client, semaphore) for u in pending]
        for i, coro in enumerate(asyncio.as_completed(tasks)):
            record = await coro
            await asyncio.sleep(DELAY_BETWEEN)
 
            if record.get("error") or not record.get("text"):
                failed.append(record["url"])
                print(f"  ❌ [{i+1}/{len(pending)}] FAILED — {record['url']}")
            else:
                succeeded.append(record)
                done.add(record["url"])
                append_jsonl(record)
                save_checkpoint(done)
                print(f"  ✅ [{i+1}/{len(pending)}] {record['title'] or record['url'][:60]}")
 
    return succeeded, failed

In [13]:
# ─── Phase 3: Playwright for JS-heavy / failed sites ─────────────────────────
 
async def scrape_playwright(url: str, page) -> dict:
    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=PLAYWRIGHT_TIMEOUT * 1000)
        await asyncio.sleep(2)  # let JS settle
        html = await page.content()
        record = parse_html(html, url)
        record["method"] = "playwright"
        record["status"] = 200
        return record
    except Exception as e:
        return {
            "url": url,
            "domain": urlparse(url).netloc,
            "title": None, "description": None, "author": None,
            "published_date": None, "keywords": None, "text": None,
            "text_length": 0,
            "scraped_at": datetime.now(timezone.utc).isoformat(),
            "method": "playwright", "status": None, "error": str(e),
        }
 
async def run_playwright_phase(urls: list[str], done: set) -> tuple[list, list]:
    pending = [u for u in urls if u not in done]
    if not pending:
        print("\n⏭️  No Playwright URLs to process.")
        return [], []
 
    try:
        from playwright.async_api import async_playwright
    except ImportError:
        print("\n⚠️  Playwright not installed. Run: pip install playwright && playwright install chromium")
        return [], pending
 
    print(f"\n🎭 Phase 3: Playwright scraping {len(pending)} URLs...")
    succeeded, failed = [], []
 
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent=HEADERS["User-Agent"],
            extra_http_headers={"Accept-Language": "en-US,en;q=0.5"}
        )
        page = await context.new_page()
 
        for i, url in enumerate(pending):
            record = await scrape_playwright(url, page)
            await asyncio.sleep(DELAY_BETWEEN)
 
            if record.get("error") or not record.get("text"):
                failed.append(url)
                print(f"  ❌ [{i+1}/{len(pending)}] FAILED — {url}")
            else:
                succeeded.append(record)
                done.add(url)
                append_jsonl(record)
                save_checkpoint(done)
                print(f"  ✅ [{i+1}/{len(pending)}] {record['title'] or url[:60]}")
 
        await browser.close()
 
    return succeeded, failed
 
 

In [14]:
# ─── Phase 4: Save to Parquet ─────────────────────────────────────────────────
 
def build_parquet():
    print("\n💾 Phase 4: Building Parquet file...")
    records = []
    if JSONL_FILE.exists():
        with JSONL_FILE.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
 
    if not records:
        print("  ⚠️  No records to save.")
        return
 
    df = pd.DataFrame(records)
 
    # Type cleanup for Parquet
    df["scraped_at"] = pd.to_datetime(df["scraped_at"], utc=True, errors="coerce")
    df["text_length"] = pd.to_numeric(df["text_length"], errors="coerce").fillna(0).astype(int)
    df["status"] = pd.to_numeric(df["status"], errors="coerce")
 
    df.to_parquet(PARQUET_FILE, index=False, compression="snappy", engine="pyarrow")
    print(f"  ✅ Saved {len(df)} rows → {PARQUET_FILE}")
    print(f"\n  Columns: {list(df.columns)}")
    print(f"  Successful: {df['error'].isna().sum()} | Failed: {df['error'].notna().sum()}")
    return df

In [16]:
# ─── Main ──────────────────────────────────────────────────────────────────────
 
async def main():
    print("=" * 60)
    print("  Social Media News Scraper")
    print(f"  {len(URLS)} URLs | Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)
 
    done = load_checkpoint()
    if done:
        print(f"\n♻️  Resuming from checkpoint ({len(done)} URLs already done)")
 
    # Phase 1: Classify
    classification = await classify_urls(URLS)
 
    # URLs to try with HTTP first (ok + blocked — Playwright may bypass blocks)
    http_candidates = classification["ok"]
    playwright_candidates = classification["blocked"] + classification["error"]
    robots_blocked = classification["robots_blocked"]
 
    # Save robots-blocked list
    if robots_blocked:
        with FAILED_FILE.open("a") as f:
            for u in robots_blocked:
                f.write(f"ROBOTS_BLOCKED\t{u}\n")
 
    # Phase 2: HTTP
    _, http_failed = await run_http_phase(http_candidates, done)
 
    # Phase 3: Playwright (blocked + http-failed)
    playwright_targets = list(set(playwright_candidates + http_failed))
    _, pw_failed = await run_playwright_phase(playwright_targets, done)
 
    # Log final failures
    if pw_failed:
        with FAILED_FILE.open("a") as f:
            for u in pw_failed:
                f.write(f"FAILED\t{u}\n")
        print(f"\n⚠️  {len(pw_failed)} URLs could not be scraped — saved to {FAILED_FILE}")
 
    # Phase 4: Parquet
    df = build_parquet()
 
    print("\n" + "=" * 60)
    print("  ✅ Done!")
    print(f"  JSONL  → {JSONL_FILE}")
    print(f"  Parquet→ {PARQUET_FILE}")
    print(f"  Failed → {FAILED_FILE}")
    print("=" * 60)
 
 
await main()

  Social Media News Scraper
  200 URLs | Started: 2026-05-10 23:14:31

📋 Phase 1: Classifying URLs...
  🚫 429 https://www.yahoo.com/entertainment/music/article/justin-bieber-is-headlining-coachella-this-month-how-the-pop-star-is-forging-a-new-path-163753630.html
  ✅ 200 https://www.theguardian.com/music/2026/apr/10/coachella-2026-justin-bieber-sabrina-carpenter-karol-g
  ✅ 200 https://www.rollingstone.com/music/music-news/justin-bieber-coachella-sneak-preview-1235540010/
  ✅ 200 https://pagesix.com/2026/04/04/celebrity-news/justin-biebers-got-a-lot-riding-on-coachella-comeback/
  ✅ 200 https://eu.desertsun.com/story/entertainment/coachella/2026/04/03/justin-bieber-loves-palm-springs-but-it-doesnt-always-love-him-back-recap-of-infamous-moments/89319620007/
  ✅ 200 https://consequence.net/2026/03/justin-bieber-private-coachella-warmup-show-no-old-songs/
  ✅ 200 https://www.tmz.com/2026/04/08/justin-bieber-coachella-rehearsal-video/
  ✅ 200 https://www.thefader.com/2026/04/08/what-will-ju